# cuGraph - GPU 加速的圖形分析

> cuGraph 提供與 NetworkX 相容的 API，讓圖形演算法在 GPU 上獲得約 39 倍的平均加速。
> 適用於社群網路分析、推薦系統、詐騙偵測等場景。

## GPU 加速系列 Notebooks

1. [Numba CUDA 基礎](./01-Introduction-to-cuda-python-with-numba.ipynb)
2. [GPU 加速概覽與環境建置](./02-GPU-Acceleration-Overview-and-Environment-Setup.ipynb)
3. [CuPy - GPU 版 NumPy](./03-CuPy-GPU-Accelerated-NumPy.ipynb)
4. [cuDF - GPU 版 pandas](./04-cuDF-GPU-Accelerated-DataFrames.ipynb)
5. [cuML - GPU 版 scikit-learn](./05-cuML-GPU-Accelerated-Machine-Learning.ipynb)
6. **[cuGraph - GPU 版 NetworkX](./06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb) ← 目前位置**
7. [Dask-CUDA 多 GPU 處理](./07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb)
8. [RAPIDS 端到端實戰](./08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb)

---
## 0. 環境檢查與安裝

In [ ]:
import shutil
import subprocess

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('未偵測到 NVIDIA GPU。')
    print('建議使用 Google Colab (免費 T4 GPU): https://colab.research.google.com')
    GPU_AVAILABLE = False

In [ ]:
# 安裝
# !pip install cugraph-cu12    # CUDA 12
# !pip install networkx        # CPU 版本（對照用）

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import time

if GPU_AVAILABLE:
    import cugraph
    import cudf
    print(f'cuGraph 版本: {cugraph.__version__}')

print(f'NetworkX 版本: {nx.__version__}')

---
## 1. 圖形分析的應用場景

| 場景 | 圖的結構 | 常用演算法 |
|------|----------|------------|
| **社群網路** | 使用者 → 節點, 好友關係 → 邊 | PageRank, 社群偵測 |
| **推薦系統** | 使用者+商品 → 節點, 購買 → 邊 | 相似度, 最短路徑 |
| **詐騙偵測** | 帳戶 → 節點, 交易 → 邊 | 異常偵測, 中心性 |
| **交通路網** | 路口 → 節點, 道路 → 邊 | 最短路徑, 連通性 |
| **知識圖譜** | 實體 → 節點, 關係 → 邊 | 子圖搜尋, 路徑分析 |

### 為什麼需要 GPU 加速？

圖形演算法通常需要遍歷大量節點和邊。當圖的規模達到數十萬節點 / 數百萬邊時，CPU 版本的 NetworkX 會變得非常慢。cuGraph 利用 GPU 的平行能力，大幅加速這些遍歷操作。

---
## 2. 建立圖形

In [ ]:
# 產生合成社群網路
# 使用 Barabási-Albert 模型（模擬真實社群網路的冪律分布）

N_nodes = 50_000
M_edges_per_node = 5  # 每個新節點連接的邊數

print(f'產生 Barabási-Albert 圖: {N_nodes:,} 節點...')
G_nx = nx.barabasi_albert_graph(N_nodes, M_edges_per_node, seed=42)
print(f'節點數: {G_nx.number_of_nodes():,}')
print(f'邊數:   {G_nx.number_of_edges():,}')

In [ ]:
if GPU_AVAILABLE:
    # 將邊列表轉為 cuDF DataFrame，再建立 cuGraph 圖
    edges = list(G_nx.edges())
    src = [e[0] for e in edges]
    dst = [e[1] for e in edges]

    edge_df = cudf.DataFrame({
        'src': src,
        'dst': dst
    })

    G_cu = cugraph.Graph()
    G_cu.from_cudf_edgelist(edge_df, source='src', destination='dst')

    print(f'cuGraph 圖已建立')
    print(f'節點數: {G_cu.number_of_vertices():,}')
    print(f'邊數:   {G_cu.number_of_edges():,}')

---
## 3. PageRank

In [ ]:
# CPU (NetworkX)
start = time.time()
pr_nx = nx.pagerank(G_nx, alpha=0.85, max_iter=100, tol=1e-6)
cpu_time = time.time() - start

# 取前 5 名
top5 = sorted(pr_nx.items(), key=lambda x: x[1], reverse=True)[:5]
print(f'NetworkX PageRank: {cpu_time:.4f} 秒')
print('Top 5 節點:')
for node, score in top5:
    print(f'  節點 {node}: {score:.6f}')

In [ ]:
# GPU (cuGraph)
if GPU_AVAILABLE:
    start = time.time()
    pr_cu = cugraph.pagerank(G_cu, alpha=0.85, max_iter=100, tol=1e-6)
    gpu_time = time.time() - start

    top5_gpu = pr_cu.nlargest(5, 'pagerank')
    print(f'cuGraph PageRank: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')
    print('\nTop 5 節點:')
    print(top5_gpu.to_pandas().to_string(index=False))
else:
    print('需要 GPU 環境。預期加速: 20-50x')

---
## 4. 社群偵測 (Louvain)

In [ ]:
# CPU (NetworkX) - 使用 community 模組
try:
    from networkx.algorithms.community import louvain_communities

    start = time.time()
    communities_nx = louvain_communities(G_nx, seed=42)
    cpu_time = time.time() - start

    print(f'NetworkX Louvain: {cpu_time:.4f} 秒')
    print(f'社群數量: {len(communities_nx)}')
    sizes = sorted([len(c) for c in communities_nx], reverse=True)
    print(f'前 5 大社群大小: {sizes[:5]}')
except Exception as e:
    print(f'NetworkX Louvain 失敗: {e}')
    cpu_time = None

In [ ]:
# GPU (cuGraph)
if GPU_AVAILABLE:
    start = time.time()
    parts_cu, modularity = cugraph.louvain(G_cu)
    gpu_time = time.time() - start

    n_communities = parts_cu['partition'].nunique()
    print(f'cuGraph Louvain: {gpu_time:.4f} 秒')
    print(f'社群數量: {n_communities}')
    print(f'模組度 (modularity): {modularity:.4f}')

    if cpu_time:
        print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

    # 各社群大小
    community_sizes = parts_cu.groupby('partition').count().sort_values(
        'vertex', ascending=False
    ).head(5).to_pandas()
    print(f'\n前 5 大社群:')
    print(community_sizes.to_string())
else:
    print('需要 GPU 環境。預期加速: 20-80x')

---
## 5. BFS 廣度優先搜尋

In [ ]:
source_node = 0  # 起始節點

# CPU (NetworkX)
start = time.time()
bfs_tree = nx.bfs_tree(G_nx, source_node)
distances_nx = nx.single_source_shortest_path_length(G_nx, source_node)
cpu_time = time.time() - start

max_depth = max(distances_nx.values())
print(f'NetworkX BFS from node {source_node}: {cpu_time:.4f} 秒')
print(f'可達節點: {len(distances_nx):,}')
print(f'最大深度: {max_depth}')

In [ ]:
# GPU (cuGraph)
if GPU_AVAILABLE:
    start = time.time()
    bfs_cu = cugraph.bfs(G_cu, source_node)
    gpu_time = time.time() - start

    max_depth_gpu = bfs_cu['distance'].max()
    reachable = (bfs_cu['distance'] < 2147483647).sum()  # 可達節點

    print(f'cuGraph BFS from node {source_node}: {gpu_time:.4f} 秒')
    print(f'可達節點: {int(reachable):,}')
    print(f'最大深度: {int(max_depth_gpu)}')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')
else:
    print('需要 GPU 環境。預期加速: 30-100x')

---
## 6. NetworkX Zero-Code-Change Backend

RAPIDS 24.10+ 推出 **`nx-cugraph`**，讓 NetworkX 自動使用 GPU 後端。

### 使用方式

```bash
# 安裝 nx-cugraph
pip install nx-cugraph-cu12
```

```python
import networkx as nx

# 方式 1：全域啟用 GPU 後端
nx.config.backend_priority = ['cugraph']  # NetworkX 3.4+

# 之後所有支援的演算法自動用 GPU
pr = nx.pagerank(G)          # 自動 GPU
communities = nx.community.louvain_communities(G)  # 自動 GPU

# 方式 2：個別指定
pr = nx.pagerank(G, backend='cugraph')
```

### 支援的演算法

- PageRank
- Betweenness Centrality
- Louvain Community Detection
- Connected Components
- Shortest Path (BFS / SSSP)
- Triangle Count
- 持續增加中...

> **最大優勢：** 完全不需要修改程式碼，也不需要學 cuGraph API。

---
## 7. 實際應用：簡易推薦系統

利用圖形分析建立「買了 A 也買了 B」的共同購買推薦。

In [ ]:
# 模擬共同購買資料
n_users = 10_000
n_products = 500
n_purchases = 100_000

np.random.seed(42)
user_ids = np.random.randint(0, n_users, n_purchases)
product_ids = np.random.randint(0, n_products, n_purchases)

# 建立共同購買邊：同一個使用者買的商品之間建立連結
from collections import defaultdict
user_purchases = defaultdict(set)
for u, p in zip(user_ids, product_ids):
    user_purchases[u].add(p)

# 產生共購邊 (取前 N 個使用者以控制圖大小)
co_purchase_edges = []
for user, products in list(user_purchases.items())[:5000]:
    products = list(products)
    for i in range(len(products)):
        for j in range(i+1, len(products)):
            co_purchase_edges.append((products[i], products[j]))

print(f'共購邊數: {len(co_purchase_edges):,}')

# NetworkX
G_copurchase = nx.Graph()
G_copurchase.add_edges_from(co_purchase_edges)
print(f'共購圖: {G_copurchase.number_of_nodes()} 商品, {G_copurchase.number_of_edges():,} 邊')

In [ ]:
# 用 PageRank 找最熱門的商品
target_product = 0  # 查詢商品 0 的推薦

# CPU 推薦
start = time.time()
pr_scores = nx.pagerank(G_copurchase, personalization={target_product: 1})
cpu_time = time.time() - start

# 排除自身，取前 5 推薦
recommendations = sorted(
    [(p, s) for p, s in pr_scores.items() if p != target_product],
    key=lambda x: x[1], reverse=True
)[:5]

print(f'商品 {target_product} 的推薦 (Personalized PageRank):')
print(f'CPU 耗時: {cpu_time:.4f} 秒')
for prod, score in recommendations:
    print(f'  商品 {prod}: 關聯分數 {score:.6f}')

In [ ]:
# GPU 推薦
if GPU_AVAILABLE:
    src_list = [e[0] for e in co_purchase_edges]
    dst_list = [e[1] for e in co_purchase_edges]

    edge_df = cudf.DataFrame({'src': src_list, 'dst': dst_list})
    G_cu_rec = cugraph.Graph()
    G_cu_rec.from_cudf_edgelist(edge_df, source='src', destination='dst')

    # cuGraph personalized PageRank
    vertices = cudf.DataFrame({'vertex': [target_product], 'values': [1.0]})

    start = time.time()
    pr_gpu = cugraph.pagerank(G_cu_rec, personalization=vertices)
    gpu_time = time.time() - start

    top5_gpu = pr_gpu[pr_gpu['vertex'] != target_product].nlargest(5, 'pagerank')
    print(f'GPU 推薦耗時: {gpu_time:.4f} 秒')
    print(f'加速倍數: {cpu_time / gpu_time:.1f}x')
    print(top5_gpu.to_pandas().to_string(index=False))

---
## 8. 效能總結

| 演算法 | 典型加速倍數 | 說明 |
|--------|-------------|------|
| PageRank | 20-50x | 迭代演算法，GPU 平行效果好 |
| Louvain 社群偵測 | 20-80x | 大圖效果更顯著 |
| BFS / SSSP | 30-100x | 圖遍歷的經典加速場景 |
| Betweenness Centrality | 50-200x | 計算量大，GPU 優勢最明顯 |
| Triangle Count | 30-100x | 平行計數 |
| Connected Components | 20-60x | 並查集平行化 |

### 經驗法則

- 節點數 < 1,000 → NetworkX 就夠快
- 節點數 1,000-10,000 → cuGraph 開始有優勢
- 節點數 > 10,000 → 強烈建議用 cuGraph
- 邊數 > 100 萬 → NetworkX 可能跑不完，cuGraph 是必要的

---
## 參考資源

- [cuGraph 官方文件](https://docs.rapids.ai/api/cugraph/stable/)
- [nx-cugraph (NetworkX GPU 後端)](https://docs.rapids.ai/api/cugraph/stable/nx_cugraph/)
- [cuGraph GitHub](https://github.com/rapidsai/cugraph)
- [NetworkX 官方文件](https://networkx.org/documentation/stable/)

> **下一步：** 前往 [07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb](./07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb) 學習多 GPU 處理。